In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Arabic Sentiment Analysis — CNN + BiLSTM Hybrid (Optimized for Small Dataset)
# ─────────────────────────────────────────────────────────────────────────────

import warnings
import random
warnings.filterwarnings("ignore")

import sys

# ✅ Fix for "RpcBackendOptions" error: reuse torch if already imported
if 'torch' not in sys.modules:
    import torch
else:
    torch = sys.modules['torch']

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# ─────────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
TRAIN_PATH   = "/content/train.csv"
VAL_PATH     = "/content/val.csv"
TEST_PATH    = "/content/test.csv"
TEXT_COL   = "clean_text"
LABEL_COL  = "labels"

TOKENIZER_ID = "aubmindlab/bert-base-arabertv2"
MAX_SEQ_LEN  = 128
NUM_LABELS   = 2

# Model hyperparameters
EMBED_DIM    = 128   # smaller embedding
CNN_FILTERS  = 64
CNN_KERNELS  = [2,3,4]
LSTM_HIDDEN  = 64
LSTM_LAYERS  = 1     # fewer layers
LSTM_DROPOUT = 0.5
MLP_HIDDEN   = 64
DROPOUT      = 0.6

# Training
BATCH_SIZE   = 32
EPOCHS       = 4
LR           = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 5.0
PATIENCE     = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. TOKENIZER
# ─────────────────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID     = tokenizer.pad_token_id
print(f"  Tokenizer vocab: {VOCAB_SIZE:,} | PAD id: {PAD_ID}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────────────
class ArabicSentimentDataset(Dataset):
    def __init__(self, df, max_len=MAX_SEQ_LEN):
        self.texts  = df[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx],
                        max_length=self.max_len,
                        padding="max_length",
                        truncation=True,
                        return_tensors="pt",
                        return_token_type_ids=False)
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

def make_loader(df, shuffle=True):
    return DataLoader(
        ArabicSentimentDataset(df),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda"
    )

# ─────────────────────────────────────────────────────────────────────────────
# 3. MODEL
# ─────────────────────────────────────────────────────────────────────────────
class CNNBranch(nn.Module):
    def __init__(self, embed_dim, n_filters, kernel_sizes):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, n_filters, k, padding=k//2) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        x = x.permute(0,2,1)  # (B, D, T)
        pooled = [F.relu(conv(x)).max(dim=2).values for conv in self.convs]
        return self.dropout(torch.cat(pooled, dim=1))

class BiLSTMAttentionBranch(nn.Module):
    def __init__(self, embed_dim, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers,
                            bidirectional=True, batch_first=True,
                            dropout=LSTM_DROPOUT if num_layers>1 else 0.0)
        self.attn_proj = nn.Linear(hidden_size*2, 1, bias=False)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x, mask):
        lstm_out, _ = self.lstm(x)
        scores = self.attn_proj(lstm_out).squeeze(-1)
        scores = scores.masked_fill(mask==0, -1e9)
        alpha = F.softmax(scores, dim=1)
        context = (alpha.unsqueeze(-1)*lstm_out).sum(dim=1)
        return self.dropout(context)

class CNNBiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels=NUM_LABELS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_ID)
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].fill_(0.0)

        self.cnn_branch = CNNBranch(EMBED_DIM, CNN_FILTERS, CNN_KERNELS)
        self.bilstm_branch = BiLSTMAttentionBranch(EMBED_DIM, LSTM_HIDDEN, LSTM_LAYERS)

        combined_dim = len(CNN_KERNELS)*CNN_FILTERS + LSTM_HIDDEN*2

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, MLP_HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN, MLP_HIDDEN//2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN//2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        emb = self.embedding(input_ids)
        cnn_out = self.cnn_branch(emb)
        lstm_out = self.bilstm_branch(emb, attention_mask)
        combined = torch.cat([cnn_out, lstm_out], dim=1)
        return self.classifier(combined)

# ─────────────────────────────────────────────────────────────────────────────
# 4. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

train_loader = make_loader(train_df)
val_loader   = make_loader(val_df, shuffle=False)
test_loader  = make_loader(test_df, shuffle=False)

# ─────────────────────────────────────────────────────────────────────────────
# 5. MODEL + OPTIMIZER + LOSS
# ─────────────────────────────────────────────────────────────────────────────
model = CNNBiLSTMClassifier(VOCAB_SIZE).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

counts = train_df[LABEL_COL].value_counts().sort_index()
weights = torch.tensor([len(train_df)/(NUM_LABELS*c) for c in counts.values], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)

# ─────────────────────────────────────────────────────────────────────────────
# 6. TRAINING LOOP WITH EARLY STOPPING
# ─────────────────────────────────────────────────────────────────────────────
best_val_f1, best_state, patience_counter = 0.0, None, 0

for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    tr_loss, preds_all, labels_all = 0, [], []
    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        tr_loss += loss.item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    scheduler.step()

    tr_f1 = f1_score(labels_all, preds_all, average="macro")
    tr_loss /= len(train_loader)

    # VALIDATION
    model.eval()
    vl_loss, vl_preds, vl_labels = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(ids, mask)
            vl_loss += criterion(logits, y).item()
            vl_preds.extend(logits.argmax(-1).cpu().numpy())
            vl_labels.extend(y.cpu().numpy())
    vl_f1 = f1_score(vl_labels, vl_preds, average="macro")
    vl_loss /= len(val_loader)

    print(f"Epoch {epoch+1} | Train Loss: {tr_loss:.4f} | F1: {tr_f1:.4f} | Val Loss: {vl_loss:.4f} | Val F1: {vl_f1:.4f}")

    # EARLY STOPPING
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_state = {k:v.clone() for k,v in model.state_dict().items()}
        torch.save(best_state, "best_cnn_bilstm.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# ─────────────────────────────────────────────────────────────────────────────
# 7. FINAL EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
model.load_state_dict(best_state)
def eval_loader(loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(ids, mask)
            preds_all.extend(logits.argmax(-1).cpu().numpy())
            labels_all.extend(y.cpu().numpy())
    return np.array(preds_all), np.array(labels_all)

vp, vl = eval_loader(val_loader)
tp, tl = eval_loader(test_loader)

print("\nValidation Set Metrics")
print(classification_report(vl, vp, target_names=["Negative","Positive"]))
print("\nTest Set Metrics")
print(classification_report(tl, tp, target_names=["Negative","Positive"]))

✅ Device: cuda


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

  Tokenizer vocab: 64,000 | PAD id: 31
Train 5,068 | Val 1,086 | Test 1,086
Epoch 1 | Train Loss: 0.6912 | F1: 0.5209 | Val Loss: 0.6718 | Val F1: 0.6148
Epoch 2 | Train Loss: 0.5664 | F1: 0.7251 | Val Loss: 0.4061 | Val F1: 0.8253
Epoch 3 | Train Loss: 0.2744 | F1: 0.9023 | Val Loss: 0.4048 | Val F1: 0.8307
Epoch 4 | Train Loss: 0.1370 | F1: 0.9629 | Val Loss: 0.4737 | Val F1: 0.8214

Validation Set Metrics
              precision    recall  f1-score   support

    Negative       0.81      0.88      0.84       554
    Positive       0.86      0.78      0.82       532

    accuracy                           0.83      1086
   macro avg       0.83      0.83      0.83      1086
weighted avg       0.83      0.83      0.83      1086


Test Set Metrics
              precision    recall  f1-score   support

    Negative       0.82      0.86      0.84       570
    Positive       0.83      0.80      0.81       516

    accuracy                           0.83      1086
   macro avg       0.83  

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Arabic Sentiment Analysis — CNN + BiLSTM Hybrid (Optimized for Small Dataset)
# ─────────────────────────────────────────────────────────────────────────────

import warnings
import random
warnings.filterwarnings("ignore")

import sys

# ✅ Fix for "RpcBackendOptions" error: reuse torch if already imported
if 'torch' not in sys.modules:
    import torch
else:
    torch = sys.modules['torch']

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# ─────────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
TRAIN_PATH   = "/content/train.csv"
VAL_PATH     = "/content/val.csv"
TEST_PATH    = "/content/test.csv"
TEXT_COL   = "clean_text"
LABEL_COL  = "labels"

TOKENIZER_ID = "aubmindlab/bert-base-arabertv2"
MAX_SEQ_LEN  = 128
NUM_LABELS   = 2

# Model hyperparameters
EMBED_DIM    = 128   # smaller embedding
CNN_FILTERS  = 64
CNN_KERNELS  = [2,3,4]
LSTM_HIDDEN  = 64
LSTM_LAYERS  = 1     # fewer layers
LSTM_DROPOUT = 0.4
MLP_HIDDEN   = 64
DROPOUT      = 0.6

# Training
BATCH_SIZE   = 32
EPOCHS       = 4
LR           = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 5.0
PATIENCE     = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. TOKENIZER
# ─────────────────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID     = tokenizer.pad_token_id
print(f"  Tokenizer vocab: {VOCAB_SIZE:,} | PAD id: {PAD_ID}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────────────
class ArabicSentimentDataset(Dataset):
    def __init__(self, df, max_len=MAX_SEQ_LEN):
        self.texts  = df[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx],
                        max_length=self.max_len,
                        padding="max_length",
                        truncation=True,
                        return_tensors="pt",
                        return_token_type_ids=False)
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

def make_loader(df, shuffle=True):
    return DataLoader(
        ArabicSentimentDataset(df),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda"
    )

# ─────────────────────────────────────────────────────────────────────────────
# 3. MODEL
# ─────────────────────────────────────────────────────────────────────────────
class CNNBranch(nn.Module):
    def __init__(self, embed_dim, n_filters, kernel_sizes):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, n_filters, k, padding=k//2) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        x = x.permute(0,2,1)  # (B, D, T)
        pooled = [F.relu(conv(x)).max(dim=2).values for conv in self.convs]
        return self.dropout(torch.cat(pooled, dim=1))

class BiLSTMAttentionBranch(nn.Module):
    def __init__(self, embed_dim, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers,
                            bidirectional=True, batch_first=True,
                            dropout=LSTM_DROPOUT if num_layers>1 else 0.0)
        self.attn_proj = nn.Linear(hidden_size*2, 1, bias=False)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x, mask):
        lstm_out, _ = self.lstm(x)
        scores = self.attn_proj(lstm_out).squeeze(-1)
        scores = scores.masked_fill(mask==0, -1e9)
        alpha = F.softmax(scores, dim=1)
        context = (alpha.unsqueeze(-1)*lstm_out).sum(dim=1)
        return self.dropout(context)

class CNNBiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels=NUM_LABELS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_ID)
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].fill_(0.0)

        self.cnn_branch = CNNBranch(EMBED_DIM, CNN_FILTERS, CNN_KERNELS)
        self.bilstm_branch = BiLSTMAttentionBranch(EMBED_DIM, LSTM_HIDDEN, LSTM_LAYERS)

        combined_dim = len(CNN_KERNELS)*CNN_FILTERS + LSTM_HIDDEN*2

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, MLP_HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN, MLP_HIDDEN//2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN//2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        emb = self.embedding(input_ids)
        cnn_out = self.cnn_branch(emb)
        lstm_out = self.bilstm_branch(emb, attention_mask)
        combined = torch.cat([cnn_out, lstm_out], dim=1)
        return self.classifier(combined)

# ─────────────────────────────────────────────────────────────────────────────
# 4. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

train_loader = make_loader(train_df)
val_loader   = make_loader(val_df, shuffle=False)
test_loader  = make_loader(test_df, shuffle=False)

# ─────────────────────────────────────────────────────────────────────────────
# 5. MODEL + OPTIMIZER + LOSS
# ─────────────────────────────────────────────────────────────────────────────
model = CNNBiLSTMClassifier(VOCAB_SIZE).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

counts = train_df[LABEL_COL].value_counts().sort_index()
weights = torch.tensor([len(train_df)/(NUM_LABELS*c) for c in counts.values], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)

# ─────────────────────────────────────────────────────────────────────────────
# 6. TRAINING LOOP WITH EARLY STOPPING
# ─────────────────────────────────────────────────────────────────────────────
best_val_f1, best_state, patience_counter = 0.0, None, 0

for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    tr_loss, preds_all, labels_all = 0, [], []
    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        tr_loss += loss.item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    scheduler.step()

    tr_f1 = f1_score(labels_all, preds_all, average="macro")
    tr_loss /= len(train_loader)

    # VALIDATION
    model.eval()
    vl_loss, vl_preds, vl_labels = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(ids, mask)
            vl_loss += criterion(logits, y).item()
            vl_preds.extend(logits.argmax(-1).cpu().numpy())
            vl_labels.extend(y.cpu().numpy())
    vl_f1 = f1_score(vl_labels, vl_preds, average="macro")
    vl_loss /= len(val_loader)

    print(f"Epoch {epoch+1} | Train Loss: {tr_loss:.4f} | F1: {tr_f1:.4f} | Val Loss: {vl_loss:.4f} | Val F1: {vl_f1:.4f}")

    # EARLY STOPPING
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_state = {k:v.clone() for k,v in model.state_dict().items()}
        torch.save(best_state, "best_cnn_bilstm.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# ─────────────────────────────────────────────────────────────────────────────
# 7. FINAL EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
model.load_state_dict(best_state)
def eval_loader(loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(ids, mask)
            preds_all.extend(logits.argmax(-1).cpu().numpy())
            labels_all.extend(y.cpu().numpy())
    return np.array(preds_all), np.array(labels_all)

vp, vl = eval_loader(val_loader)
tp, tl = eval_loader(test_loader)

print("\nValidation Set Metrics")
print(classification_report(vl, vp, target_names=["Negative","Positive"]))
print("\nTest Set Metrics")
print(classification_report(tl, tp, target_names=["Negative","Positive"]))

✅ Device: cuda
  Tokenizer vocab: 64,000 | PAD id: 31
Train 5,068 | Val 1,086 | Test 1,086
Epoch 1 | Train Loss: 0.6912 | F1: 0.5209 | Val Loss: 0.6718 | Val F1: 0.6148
Epoch 2 | Train Loss: 0.5664 | F1: 0.7251 | Val Loss: 0.4061 | Val F1: 0.8253
Epoch 3 | Train Loss: 0.2744 | F1: 0.9023 | Val Loss: 0.4048 | Val F1: 0.8307
Epoch 4 | Train Loss: 0.1370 | F1: 0.9629 | Val Loss: 0.4737 | Val F1: 0.8214

Validation Set Metrics
              precision    recall  f1-score   support

    Negative       0.81      0.88      0.84       554
    Positive       0.86      0.78      0.82       532

    accuracy                           0.83      1086
   macro avg       0.83      0.83      0.83      1086
weighted avg       0.83      0.83      0.83      1086


Test Set Metrics
              precision    recall  f1-score   support

    Negative       0.82      0.86      0.84       570
    Positive       0.83      0.80      0.81       516

    accuracy                           0.83      1086
   macro a